In [ ]:
%pip -q install datasets safetensors

In [ ]:
import gc
import json
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch import nn
from torch.optim import Adam
from pathlib import Path
from datasets import load_dataset
from safetensors.torch import load_file

ROOT = Path("/content/drive/MyDrive/LoRe")
PLLM_DIR = ROOT / "PersonalLLM_embeddings"

if "train_ds" not in globals() or "test_ds" not in globals():
    dataset = load_dataset("namkoong-lab/PersonalLLM")
    train_ds = dataset["train"]
    test_ds = dataset["test"]

if "train_emb" not in globals():
    train_emb = load_file(str(PLLM_DIR / "train.safetensors"))["embeddings"]

if "test_emb" not in globals():
    test_emb = load_file(str(PLLM_DIR / "test.safetensors"))["embeddings"]

D = 4096
N_RESPONSES = 8

# Support either flat cached embeddings or already-reshaped embeddings.
if train_emb.ndim == 2:
    train_emb = train_emb.reshape(len(train_ds), N_RESPONSES, D)

if test_emb.ndim == 2:
    test_emb = test_emb.reshape(len(test_ds), N_RESPONSES, D)

train_emb = train_emb.cpu().contiguous()
test_emb = test_emb.cpu().contiguous()

MODEL_NAMES = [
    "gemma_2b",
    "gemma_7b",
    "mistral_raft",
    "llama3_sfairx",
    "oasst_deberta_v3",
    "beaver_7b",
    "oasst_pythia_7b",
    "oasst_pythia_1b",
    "mistral_ray",
    "mistral_weqweasdas",
]

def build_reward_scores(split):
    scores = np.empty(
        (len(split), N_RESPONSES, len(MODEL_NAMES)),
        dtype=np.float32,
    )

    for response_idx in range(1, N_RESPONSES + 1):
        for model_idx, model_name in enumerate(MODEL_NAMES):
            column = f"response_{response_idx}_{model_name}"
            scores[:, response_idx - 1, model_idx] = np.asarray(
                split[column], dtype=np.float32
            )

    if not np.isfinite(scores).all():
        raise ValueError("Reward scores contain non-finite values.")

    return scores

reward_train = build_reward_scores(train_ds)
reward_test = build_reward_scores(test_ds)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Training embeddings:", tuple(train_emb.shape), train_emb.dtype)
print("Test embeddings:", tuple(test_emb.shape), test_emb.dtype)
print("Training reward scores:", reward_train.shape)
print("Test reward scores:", reward_test.shape)

assert train_emb.shape == (9402, 8, 4096)
assert test_emb.shape == (1000, 8, 4096)
assert reward_train.shape == (9402, 8, 10)
assert reward_test.shape == (1000, 8, 10)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Training embeddings: (9402, 8, 4096) torch.bfloat16
Test embeddings: (1000, 8, 4096) torch.bfloat16
Training reward scores: (9402, 8, 10)
Test reward scores: (1000, 8, 10)


In [ ]:
def build_user_features(scores, embeddings, user_weights, n_examples, rng):
    """
    For each synthetic user, sample training prompts and construct
    embedding differences: e(x, y_chosen) - e(x, y_rejected).

    Returns:
        X: [num_users, n_examples, D], float32 CPU tensor
        selected_indices: [num_users, n_examples]
    """
    n_prompts = scores.shape[0]
    n_users = len(user_weights)
    n_examples = min(n_examples, n_prompts)

    X = torch.empty((n_users, n_examples, D), dtype=torch.float32)
    selected_indices = np.empty((n_users, n_examples), dtype=np.int32)

    for i, w in enumerate(user_weights):
        prompt_ids = rng.choice(
            n_prompts, size=n_examples, replace=False
        )
        selected_indices[i] = prompt_ids

        # Score all eight responses under this synthetic user's preferences.
        user_scores = np.einsum(
            "prd,d->pr", scores[prompt_ids], w
        )
        chosen = user_scores.argmax(axis=1)
        rejected = user_scores.argmin(axis=1)

        p = torch.as_tensor(prompt_ids, dtype=torch.long)
        c = torch.as_tensor(chosen, dtype=torch.long)
        r = torch.as_tensor(rejected, dtype=torch.long)

        # Subtract in the cached embedding dtype, then cast to float32.
        diff = embeddings[p, c] - embeddings[p, r]
        X[i] = diff.float()

    return X, selected_indices


def fit_joint_lore(X_seen, K, steps=2000, lr=0.5, log_every=500):
    """Jointly optimize the shared basis V and seen-user weight logits."""
    Xg = X_seen.to(device=device, dtype=torch.float32)
    n_users, n_examples, d = Xg.shape

    # Same parameter shapes and initialization family as our earlier notebook.
    V = nn.Parameter(torch.randn(d, K, device=device))
    W_logits = nn.Parameter(torch.randn(n_users, K, device=device))

    optimizer = Adam([V, W_logits], lr=lr)
    losses = []

    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)

        W = torch.softmax(W_logits, dim=1)

        projected = (Xg.reshape(-1, d) @ V).reshape(
            n_users, n_examples, K
        )
        margins = (projected * W[:, None, :]).sum(dim=-1) / 100.0

        # Mean over examples for each user, summed across users.
        loss = F.softplus(-margins).mean(dim=1).sum()

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite joint loss at step {step + 1}")

        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

        if (step + 1) % log_every == 0:
            print(
                f"  Joint {step + 1:4d}/{steps} | "
                f"loss={loss.item():.4f}"
            )

    V_cpu = V.detach().cpu()
    W_cpu = torch.softmax(W_logits.detach(), dim=1).cpu()

    del Xg, V, W_logits, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return V_cpu, W_cpu, losses


def fit_fewshot_lore(X_fewshot, V_fixed, K, steps=1000, lr=0.1, log_every=250):
    """Freeze V and optimize only unseen-user weight logits."""
    Xg = X_fewshot.to(device=device, dtype=torch.float32)
    Vg = V_fixed.to(device=device, dtype=torch.float32).detach()

    n_users, n_examples, d = Xg.shape
    W_logits = nn.Parameter(torch.randn(n_users, K, device=device))
    optimizer = Adam([W_logits], lr=lr)
    losses = []

    for step in range(steps):
        optimizer.zero_grad(set_to_none=True)

        W = torch.softmax(W_logits, dim=1)

        projected = (Xg.reshape(-1, d) @ Vg).reshape(
            n_users, n_examples, K
        )
        margins = (projected * W[:, None, :]).sum(dim=-1) / 100.0

        loss = F.softplus(-margins).mean(dim=1).sum()

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite few-shot loss at step {step + 1}")

        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

        if (step + 1) % log_every == 0:
            print(
                f"  Few-shot {step + 1:4d}/{steps} | "
                f"loss={loss.item():.4f}"
            )

    W_cpu = torch.softmax(W_logits.detach(), dim=1).cpu()

    del Xg, Vg, W_logits, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return W_cpu, losses


@torch.no_grad()
def accuracy_on_prebuilt_features(X, fitted_weights, V):
    """Accuracy on an already prepared set of chosen/rejected pairs."""
    Xg = X.to(device=device, dtype=torch.float32)
    Wg = torch.as_tensor(
        fitted_weights, dtype=torch.float32, device=device
    )
    Vg = V.to(device=device, dtype=torch.float32)

    n_users, n_examples, d = Xg.shape
    K = Wg.shape[1]

    projected = (Xg.reshape(-1, d) @ Vg).reshape(
        n_users, n_examples, K
    )
    margins = (projected * Wg[:, None, :]).sum(dim=-1)

    return (margins > 0).float().mean(dim=1).cpu().numpy()


@torch.no_grad()
def evaluate_population(
    true_user_weights,
    fitted_weights,
    reward_scores,
    embeddings,
    V,
    user_batch_size=8,
):
    """
    Evaluate on every test prompt by generating chosen/rejected labels
    from each synthetic user's true reward-model mixture.
    """
    scores_gpu = torch.as_tensor(
        reward_scores, dtype=torch.float32, device=device
    )
    emb_gpu = embeddings.to(device)
    W_true = torch.as_tensor(
        true_user_weights, dtype=torch.float32, device=device
    )
    W_fit = torch.as_tensor(
        fitted_weights, dtype=torch.float32, device=device
    )
    Vg = V.to(device=device, dtype=torch.float32)

    n_users = len(true_user_weights)
    n_prompts = reward_scores.shape[0]
    K = Vg.shape[1]
    all_user_accuracies = []

    prompt_grid = torch.arange(n_prompts, device=device).unsqueeze(0)

    for start in range(0, n_users, user_batch_size):
        end = min(start + user_batch_size, n_users)

        # [users in batch, prompts, 8 candidate responses]
        candidate_scores = torch.einsum(
            "prm,um->upr",
            scores_gpu,
            W_true[start:end],
        )

        chosen_idx = candidate_scores.argmax(dim=-1)
        rejected_idx = candidate_scores.argmin(dim=-1)

        prompt_idx = prompt_grid.expand(end - start, -1)
        chosen_emb = emb_gpu[prompt_idx, chosen_idx]
        rejected_emb = emb_gpu[prompt_idx, rejected_idx]

        differences = (chosen_emb - rejected_emb).float()

        projected = (differences.reshape(-1, D) @ Vg).reshape(
            end - start, n_prompts, K
        )
        margins = (
            projected * W_fit[start:end, None, :]
        ).sum(dim=-1)

        batch_accuracy = (margins > 0).float().mean(dim=1)
        all_user_accuracies.extend(batch_accuracy.cpu().tolist())

        del candidate_scores, chosen_emb, rejected_emb, differences

    return np.asarray(all_user_accuracies, dtype=np.float64)

In [ ]:
# ---------------- Experiment configuration ----------------
K = 5

SEEDS = [42, 43, 44, 45, 46]
DIRICHLET_ALPHAS = [0.1, 0.001]

N_SEEN = 1000
N_UNSEEN = 500

N_SEEN_PAIRS = int(0.005 * len(train_ds))     # 47
N_FEWSHOT_PAIRS = int(0.001 * len(train_ds))  # 9

JOINT_STEPS = 2000
FEWSHOT_STEPS = 1000
JOINT_LR = 0.5
FEWSHOT_LR = 0.1

# Reference-model regularization remains disabled.
REG_ALPHA = 0.0

SWEEP_DIR = (
    ROOT / "PersonalLLM_runs"
    / "K5_alpha_sweep_5seeds_joint2000_fewshot1000"
)
SWEEP_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = SWEEP_DIR / "run_metrics.csv"

if RESULTS_CSV.exists():
    run_df = pd.read_csv(RESULTS_CSV)
else:
    run_df = pd.DataFrame()

print("Seeds:", SEEDS)
print("Dirichlet alphas:", DIRICHLET_ALPHAS)
print("K:", K)
print("Seen users:", N_SEEN)
print("Unseen users:", N_UNSEEN)
print("Seen examples/user:", N_SEEN_PAIRS)
print("Few-shot examples/unseen user:", N_FEWSHOT_PAIRS)
print("Joint steps:", JOINT_STEPS)
print("Few-shot steps:", FEWSHOT_STEPS)
print("Output directory:", SWEEP_DIR)


def alpha_tag(alpha):
    return str(alpha).replace(".", "p")


for alpha_user in DIRICHLET_ALPHAS:
    for seed in SEEDS:
        run_id = f"alpha_{alpha_tag(alpha_user)}_seed_{seed}"
        checkpoint_path = SWEEP_DIR / f"{run_id}.pt"

        already_recorded = (
            not run_df.empty
            and "run_id" in run_df.columns
            and run_id in set(run_df["run_id"].astype(str))
        )

        if already_recorded and checkpoint_path.exists():
            print(f"\nSkipping completed run: {run_id}")
            continue

        print("\n" + "=" * 72)
        print(f"Run: {run_id}")
        print("=" * 72)

        start_time = time.time()

        # Every run gets an independent synthetic population and model seed.
        rng = np.random.default_rng(seed)
        random.seed(seed)
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        dirichlet_vector = np.full(
            reward_train.shape[-1], alpha_user, dtype=np.float64
        )

        W_seen_true = rng.dirichlet(
            dirichlet_vector, size=N_SEEN
        )
        W_unseen_true = rng.dirichlet(
            dirichlet_vector, size=N_UNSEEN
        )

        # Generate training pairs from the synthetic users.
        X_seen, seen_prompt_ids = build_user_features(
            reward_train,
            train_emb,
            W_seen_true,
            N_SEEN_PAIRS,
            rng,
        )

        X_fewshot, fewshot_prompt_ids = build_user_features(
            reward_train,
            train_emb,
            W_unseen_true,
            N_FEWSHOT_PAIRS,
            rng,
        )

        print("Seen feature tensor:", tuple(X_seen.shape))
        print("Few-shot feature tensor:", tuple(X_fewshot.shape))

        # Stage 1: jointly learn the shared basis and seen-user weights.
        V_cpu, W_seen_fit, joint_losses = fit_joint_lore(
            X_seen,
            K=K,
            steps=JOINT_STEPS,
            lr=JOINT_LR,
        )

        # Stage 2: freeze the basis; adapt unseen-user weights.
        W_unseen_fit, fewshot_losses = fit_fewshot_lore(
            X_fewshot,
            V_fixed=V_cpu,
            K=K,
            steps=FEWSHOT_STEPS,
            lr=FEWSHOT_LR,
        )

        # Training-set diagnostics.
        seen_train_user_acc = accuracy_on_prebuilt_features(
            X_seen, W_seen_fit, V_cpu
        )
        fewshot_train_user_acc = accuracy_on_prebuilt_features(
            X_fewshot, W_unseen_fit, V_cpu
        )

        # Primary metrics: unseen prompts for seen and unseen users.
        seen_test_user_acc = evaluate_population(
            W_seen_true,
            W_seen_fit,
            reward_test,
            test_emb,
            V_cpu,
        )

        unseen_test_user_acc = evaluate_population(
            W_unseen_true,
            W_unseen_fit,
            reward_test,
            test_emb,
            V_cpu,
        )

        # Each run's metrics are macro-averages across its users.
        record = {
            "run_id": run_id,
            "seed": seed,
            "dirichlet_alpha": alpha_user,
            "K": K,
            "n_seen_users": N_SEEN,
            "n_unseen_users": N_UNSEEN,
            "seen_pairs_per_user": N_SEEN_PAIRS,
            "fewshot_pairs_per_user": N_FEWSHOT_PAIRS,
            "joint_steps": JOINT_STEPS,
            "fewshot_steps": FEWSHOT_STEPS,
            "regularization_alpha": REG_ALPHA,

            "seen_train_mean_pct": 100 * seen_train_user_acc.mean(),
            "seen_train_user_sd_pct": 100 * seen_train_user_acc.std(),

            "seen_test_mean_pct": 100 * seen_test_user_acc.mean(),
            "seen_test_user_sd_pct": 100 * seen_test_user_acc.std(),

            "fewshot_train_mean_pct": 100 * fewshot_train_user_acc.mean(),
            "fewshot_train_user_sd_pct": 100 * fewshot_train_user_acc.std(),

            "unseen_test_mean_pct": 100 * unseen_test_user_acc.mean(),
            "unseen_test_user_sd_pct": 100 * unseen_test_user_acc.std(),

            "runtime_seconds": time.time() - start_time,
            "final_joint_loss": joint_losses[-1],
            "final_fewshot_loss": fewshot_losses[-1],
        }

        # Save a model checkpoint for this exact run.
        torch.save(
            {
                "config": record,
                "V": V_cpu,
                "W_seen_fit": W_seen_fit,
                "W_unseen_fit": W_unseen_fit,
                "W_seen_true": W_seen_true,
                "W_unseen_true": W_unseen_true,
                "seen_prompt_indices": seen_prompt_ids,
                "fewshot_prompt_indices": fewshot_prompt_ids,
                "joint_losses": joint_losses,
                "fewshot_losses": fewshot_losses,
            },
            checkpoint_path,
        )

        # Update the CSV after each run for resumability.
        if not run_df.empty and "run_id" in run_df.columns:
            run_df = run_df[run_df["run_id"] != run_id]

        run_df = pd.concat(
            [run_df, pd.DataFrame([record])],
            ignore_index=True,
        )
        run_df.to_csv(RESULTS_CSV, index=False)

        print(f"\nCompleted: {run_id}")
        print(f"Seen-user test accuracy: {record['seen_test_mean_pct']:.2f}%")
        print(f"Unseen-user test accuracy: {record['unseen_test_mean_pct']:.2f}%")
        print(f"Saved checkpoint: {checkpoint_path}")
        print(f"Updated results: {RESULTS_CSV}")

        del X_seen, X_fewshot
        del W_seen_true, W_unseen_true, W_seen_fit, W_unseen_fit
        del V_cpu, joint_losses, fewshot_losses
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("\nSweep loop finished.")
print("Results CSV:", RESULTS_CSV)

Seeds: [42, 43, 44, 45, 46]
Dirichlet alphas: [0.1, 0.001]
K: 5
Seen users: 1000
Unseen users: 500
Seen examples/user: 47
Few-shot examples/unseen user: 9
Joint steps: 2000
Few-shot steps: 1000
Output directory: /content/drive/MyDrive/LoRe/PersonalLLM_runs/K5_alpha_sweep_5seeds_joint2000_fewshot1000

Run: alpha_0p1_seed_42
Seen feature tensor: (1000, 47, 4096)
Few-shot feature tensor: (500, 9, 4096)
  Joint  500/2000 | loss=48.4436
  Joint 1000/2000 | loss=29.5559
  Joint 1500/2000 | loss=18.9949
  Joint 2000/2000 | loss=12.2330
  Few-shot  250/1000 | loss=16.6928
  Few-shot  500/1000 | loss=16.3877
  Few-shot  750/1000 | loss=16.3203
  Few-shot 1000/1000 | loss=16.2915

Completed: alpha_0p1_seed_42
Seen-user test accuracy: 95.30%
Unseen-user test accuracy: 94.22%
Saved checkpoint: /content/drive/MyDrive/LoRe/PersonalLLM_runs/K5_alpha_sweep_5seeds_joint2000_fewshot1000/alpha_0p1_seed_42.pt
Updated results: /content/drive/MyDrive/LoRe/PersonalLLM_runs/K5_alpha_sweep_5seeds_joint2000_few

In [ ]:
import pandas as pd
import numpy as np

run_df = pd.read_csv(RESULTS_CSV)

METRICS = {
    "Seen-user test": "seen_test_mean_pct",
    "Unseen-user test": "unseen_test_mean_pct",
    "Seen training (diagnostic)": "seen_train_mean_pct",
    "Few-shot training (diagnostic)": "fewshot_train_mean_pct",
}

summary_rows = []

for alpha_user in DIRICHLET_ALPHAS:
    group = run_df[
        np.isclose(run_df["dirichlet_alpha"], alpha_user)
    ].sort_values("seed")

    print(f"\nDirichlet alpha = {alpha_user}")
    print(f"Completed runs: {len(group)}/{len(SEEDS)}")

    for metric_name, col in METRICS.items():
        mean = group[col].mean()
        sd = group[col].std(ddof=1) if len(group) > 1 else np.nan

        print(f"{metric_name:32s}: {mean:.2f}% ± {sd:.2f}%")

        summary_rows.append({
            "dirichlet_alpha": alpha_user,
            "metric": metric_name,
            "mean_pct": mean,
            "std_across_seeds_pct": sd,
            "num_runs": len(group),
        })

summary_df = pd.DataFrame(summary_rows)

SUMMARY_CSV = SWEEP_DIR / "five_seed_summary.csv"
summary_df.to_csv(SUMMARY_CSV, index=False)

print("\nSaved summary:", SUMMARY_CSV)


Dirichlet alpha = 0.1
Completed runs: 5/5
Seen-user test                  : 95.63% ± 0.20%
Unseen-user test                : 94.46% ± 0.41%
Seen training (diagnostic)      : 99.95% ± 0.07%
Few-shot training (diagnostic)  : 98.93% ± 0.07%

Dirichlet alpha = 0.001
Completed runs: 5/5
Seen-user test                  : 93.90% ± 0.24%
Unseen-user test                : 92.43% ± 0.63%
Seen training (diagnostic)      : 99.90% ± 0.09%
Few-shot training (diagnostic)  : 98.43% ± 0.19%

Saved summary: /content/drive/MyDrive/LoRe/PersonalLLM_runs/K5_alpha_sweep_5seeds_joint2000_fewshot1000/five_seed_summary.csv


In [ ]:
import numpy as np
import pandas as pd

run_df = pd.read_csv(RESULTS_CSV)

# Published values from LoRe, Table 1 (PersonalLLM).
# The reported +/- values are across 20 experiment repetitions.
PAPER_RESULTS = {
    0.1: {
        "seen_test_mean_pct": (96.0, 0.7),
        "unseen_test_mean_pct": (96.1, 0.8),
    },
    0.001: {
        "seen_test_mean_pct": (94.3, 0.9),
        "unseen_test_mean_pct": (93.3, 1.0),
    },
}

comparison_rows = []

for alpha_user, paper_metrics in PAPER_RESULTS.items():
    group = run_df[
        np.isclose(run_df["dirichlet_alpha"], alpha_user)
    ]

    for column, paper_values in paper_metrics.items():
        paper_mean, paper_sd = paper_values

        ours_mean = group[column].mean() if len(group) else np.nan
        ours_sd = (
            group[column].std(ddof=1)
            if len(group) > 1
            else np.nan
        )

        comparison_rows.append({
            "Dirichlet alpha": alpha_user,
            "Metric": (
                "Seen-user test"
                if column == "seen_test_mean_pct"
                else "Unseen-user test"
            ),
            "Paper mean (%)": paper_mean,
            "Paper SD (%)": paper_sd,
            "Our mean (%)": ours_mean,
            "Our SD across 5 seeds (%)": ours_sd,
            "Mean difference (pp)": ours_mean - paper_mean,
            "Runs completed": len(group),
        })

comparison_df = pd.DataFrame(comparison_rows)

print("Paper vs. our five-seed experiment")
display(
    comparison_df.round(2)
)

comparison_df.to_csv(
    SWEEP_DIR / "paper_vs_ours.csv",
    index=False,
)

print("Saved:", SWEEP_DIR / "paper_vs_ours.csv")

Paper vs. our five-seed experiment


,Dirichlet alpha,Metric,Paper mean (%),Paper SD (%),Our mean (%),Our SD across 5 seeds (%),Mean difference (pp),Runs completed
0,0.1,Seen-user test,96.0,0.7,95.63,0.20,-0.37,5
1,0.1,Unseen-user test,96.1,0.8,94.46,0.41,-1.64,5
2,0.0,Seen-user test,94.3,0.9,93.90,0.24,-0.40,5
3,0.0,Unseen-user test,93.3,1.0,92.43,0.63,-0.87,5


Saved: /content/drive/MyDrive/LoRe/PersonalLLM_runs/K5_alpha_sweep_5seeds_joint2000_fewshot1000/paper_vs_ours.csv


In [ ]:
import pickle
import json
from pathlib import Path
import numpy as np

BASE = Path("/content/drive/MyDrive/LoRe")
REDDIT_DIR = BASE / "RedditTLDR_embeddings"

with open(REDDIT_DIR / "tldr_embeddings_train.pkl", "rb") as f:
    train_by_user = pickle.load(f)

with open(REDDIT_DIR / "tldr_embeddings_val.pkl", "rb") as f:
    val_by_user = pickle.load(f)

with open(REDDIT_DIR / "user_split.json", "r") as f:
    user_split = json.load(f)

seen_users = set(user_split["seen_workers"])
unseen_users = set(user_split["unseen_workers"])

print("=" * 65)
print("DATASET COUNTS")
print("=" * 65)

print("Training workers:", len(train_by_user))
print("Validation workers:", len(val_by_user))
print("Seen workers:", len(seen_users))
print("Unseen workers:", len(unseen_users))

print("Training comparisons:", sum(len(v) for v in train_by_user.values()))
print("Validation comparisons:", sum(len(v) for v in val_by_user.values()))

print("\nWorkers missing from training:",
      sorted((seen_users | unseen_users) - set(train_by_user)))
print("Workers missing from validation:",
      sorted((seen_users | unseen_users) - set(val_by_user)))

print("\n" + "=" * 65)
print("TRAINING COMPARISONS PER WORKER")
print("=" * 65)

for group_name, workers in [
    ("SEEN", seen_users),
    ("UNSEEN", unseen_users),
]:
    counts = {
        worker: len(train_by_user.get(worker, []))
        for worker in sorted(workers)
    }

    print(f"\n{group_name} workers:")
    print(counts)
    if counts:
        vals = list(counts.values())
        print(
            f"Count range: {min(vals)}–{max(vals)}; "
            f"mean: {np.mean(vals):.2f}"
        )


# Recursively show types and shapes without dumping tensor values.
def describe(obj, path="root", depth=0, max_depth=3):
    indent = "  " * depth

    if hasattr(obj, "shape"):
        print(f"{indent}{path}: {type(obj).__name__}, shape={obj.shape}")
        return

    if isinstance(obj, dict):
        print(f"{indent}{path}: dict, keys={list(obj.keys())}")
        if depth < max_depth:
            for key, value in obj.items():
                describe(value, str(key), depth + 1, max_depth)
        return

    if isinstance(obj, (list, tuple)):
        print(f"{indent}{path}: {type(obj).__name__}, len={len(obj)}")
        if obj and depth < max_depth:
            describe(obj[0], "[0]", depth + 1, max_depth)
        return

    if isinstance(obj, (str, int, float, bool)) or obj is None:
        print(f"{indent}{path}: {type(obj).__name__}, preview={repr(obj)[:100]}")
        return

    print(f"{indent}{path}: {type(obj).__name__}")


print("\n" + "=" * 65)
print("ONE TRAINING RECORD")
print("=" * 65)

sample_worker = next(iter(train_by_user))
sample_record = train_by_user[sample_worker][0]

print("Worker ID:", sample_worker)
print("Record keys:", list(sample_record.keys()))

for key, value in sample_record.items():
    if key == "embeddings":
        describe(value, path="embeddings")
    elif isinstance(value, str):
        print(f"{key}: {value[:150]!r}")
    else:
        describe(value, path=key, max_depth=1)


print("\n" + "=" * 65)
print("ONE VALIDATION RECORD")
print("=" * 65)

val_worker = next(iter(val_by_user))
val_record = val_by_user[val_worker][0]

print("Worker ID:", val_worker)
print("Record keys:", list(val_record.keys()))
describe(val_record["embeddings"], path="embeddings")

DATASET COUNTS
Training workers: 39
Validation workers: 39
Seen workers: 20
Unseen workers: 19
Training comparisons: 3865
Validation comparisons: 10275

Workers missing from training: []
Workers missing from validation: []

TRAINING COMPARISONS PER WORKER

SEEN workers:
{'3AFaFd3w9NjDGnO51kupLyK1N44DQ2': 150, '44Z8ttpKcY6Kr1sNymNnBA0nL0h4dZ': 84, 'D8z53gLFLFqhZowaegbtxmSGa0jqv0': 150, 'HNzkrs9geGu1YMMfZ5Qvdt0ZaCthfB': 150, 'ISkqTUKflxjg9CBXeghrNug5YTZ3zS': 150, 'Jxv4hxfb9zTVa5nsMDFlnjSX5LZ8MK': 150, 'KZL1qeRzHNYSfDAuOctL1iyVV8WC5N': 150, 'M3icahkfAtC9CJrtKgQ7qvyZ5SD8wC': 150, 'M4bdOszgybjO2qg2Dth5I1GOYAvE7V': 150, 'P2p07Up4eJyvxrrVYgwtb60krbFbxI': 150, 'V4kZVolIcM13xFttt12rzhEn7krHK4': 131, 'XxIBEfyZigAJyqYeRnjaZsq4ADwPmW': 150, 'ZzGCcAhvqF0HnKxNsUjtJFadcZdyZj': 150, 'a7zXgbkuY6lk3vdt0q2Qf7SrZQgZ86': 150, 'gMlGeJl1vsMERrbmC7W717zpVevUh8': 150, 'iL7GfrbN2PeB3KInidqSxUdxYcTZmG': 150, 'rmgbTjW1stlproQnuHE2bUpK78Jxle': 150, 'thott7XepukYSbOL2QgSlyXd0rgHvr': 150, 'uvzut5OK2bvei9zoCDdktcfLEN

In [ ]:
import torch
import numpy as np

# --------------------------------------------------
# Convert each cached comparison into:
# winning embedding - losing embedding
# --------------------------------------------------
def records_to_deltas(records, worker_id, split_name):
    deltas = []

    for idx, record in enumerate(records):
        try:
            winning = torch.as_tensor(
                record["embeddings"]["winning"][0]
            ).detach().cpu().float().reshape(-1)

            losing = torch.as_tensor(
                record["embeddings"]["losing"][0]
            ).detach().cpu().float().reshape(-1)

        except Exception as e:
            raise ValueError(
                f"Could not read {split_name} record {idx} "
                f"for worker {worker_id}"
            ) from e

        if winning.shape != losing.shape:
            raise ValueError(
                f"Embedding shape mismatch for worker {worker_id}, "
                f"record {idx}: {winning.shape} vs {losing.shape}"
            )

        if winning.numel() != 4096:
            raise ValueError(
                f"Expected 4096 features, got {winning.numel()} "
                f"for worker {worker_id}, record {idx}"
            )

        deltas.append(winning - losing)

    if not deltas:
        raise ValueError(
            f"No comparisons found for worker {worker_id} in {split_name}"
        )

    return torch.stack(deltas, dim=0)


# Build CPU tensors for every worker.
train_diffs = {
    worker: records_to_deltas(records, worker, "train")
    for worker, records in train_by_user.items()
}

val_diffs = {
    worker: records_to_deltas(records, worker, "validation")
    for worker, records in val_by_user.items()
}


# --------------------------------------------------
# Sanity checks
# --------------------------------------------------
assert set(train_diffs) == seen_users | unseen_users
assert set(val_diffs) == seen_users | unseen_users

assert all(x.ndim == 2 and x.shape[1] == 4096
           for x in train_diffs.values())

assert all(x.ndim == 2 and x.shape[1] == 4096
           for x in val_diffs.values())

assert all(torch.isfinite(x).all().item()
           for x in train_diffs.values())

assert all(torch.isfinite(x).all().item()
           for x in val_diffs.values())


print("Feature preparation successful.")
print("Feature dimension:", next(iter(train_diffs.values())).shape[1])
print("Training comparisons:", sum(len(x) for x in train_diffs.values()))
print("Validation comparisons:", sum(len(x) for x in val_diffs.values()))

print("\nSeen-user comparison counts:")
print([len(train_diffs[u]) for u in sorted(seen_users)])

print("\nUnseen-user few-shot counts:")
print([len(train_diffs[u]) for u in sorted(unseen_users)])

print("\nExample tensor shape:", train_diffs[next(iter(train_diffs))].shape)
print("Training tensor device:", train_diffs[next(iter(train_diffs))].device)
print("All embeddings are finite.")

Feature preparation successful.
Feature dimension: 4096
Training comparisons: 3865
Validation comparisons: 10275

Seen-user comparison counts:
[150, 84, 150, 150, 150, 150, 150, 150, 150, 150, 131, 150, 150, 150, 150, 150, 150, 150, 150, 150]

Unseen-user few-shot counts:
[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]

Example tensor shape: torch.Size([50, 4096])
Training tensor device: cpu
All embeddings are finite.


In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------------------------------------
# Experiment configuration
# --------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

K = 5
JOINT_STEPS = 1000
JOINT_LR = 0.5
FEWSHOT_STEPS = 500
FEWSHOT_LR = 0.1
LOGIT_SCALE = 100.0

print("Device:", DEVICE)
print("K:", K)
print("Joint training:", JOINT_STEPS, "steps; lr =", JOINT_LR)
print("Few-shot adaptation:", FEWSHOT_STEPS, "steps; lr =", FEWSHOT_LR)


def set_seed(seed):
    """Set random seeds for reproducible experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def pack_user_diffs(diffs_by_user, workers):
    """
    Concatenate per-user preference differences.

    Returns:
        X_cat: [total_comparisons, 4096]
        user_idx: user index for each comparison
        slices: slices identifying each user's comparisons
        worker_order: stable ordering of workers
    """
    worker_order = list(workers)

    if not worker_order:
        raise ValueError("No workers supplied.")

    xs = []
    user_indices = []
    slices = []
    start = 0

    for user_idx, worker in enumerate(worker_order):
        if worker not in diffs_by_user:
            raise KeyError(f"Missing embeddings for worker: {worker}")

        x = diffs_by_user[worker]

        if x.ndim != 2 or x.shape[1] != 4096:
            raise ValueError(
                f"Expected [n, 4096] for {worker}, got {tuple(x.shape)}"
            )

        if len(x) == 0:
            raise ValueError(f"Worker {worker} has no comparisons.")

        x = x.to(device=DEVICE, dtype=torch.float32)
        xs.append(x)

        n = len(x)
        user_indices.append(
            torch.full((n,), user_idx, dtype=torch.long, device=DEVICE)
        )
        slices.append(slice(start, start + n))
        start += n

    X_cat = torch.cat(xs, dim=0)
    user_idx_tensor = torch.cat(user_indices, dim=0)

    return X_cat, user_idx_tensor, slices, worker_order


# --------------------------------------------------
# Jointly learn shared basis V and seen-user weights W
# --------------------------------------------------
def fit_lore_joint(diffs_by_user, workers, seed,
                   k=K, steps=JOINT_STEPS, lr=JOINT_LR):
    """
    LoRe joint training with alpha=0 (unregularized basis).

    V: [4096, k] shared reward basis
    W: [number_of_seen_users, k] user-specific logits
    softmax(W): user-specific simplex weights
    """
    set_seed(seed)

    X_cat, user_idx, slices, worker_order = pack_user_diffs(
        diffs_by_user, workers
    )

    n_users = len(worker_order)
    n_features = X_cat.shape[1]

    W = nn.Parameter(torch.randn(n_users, k, device=DEVICE))
    V = nn.Parameter(torch.randn(n_features, k, device=DEVICE))

    optimizer = torch.optim.Adam([W, V], lr=lr)

    for step in range(steps):
        optimizer.zero_grad()

        # Project each preference difference into the shared basis.
        projected = X_cat @ V

        # Each comparison is scored using its own user's mixture weights.
        user_weights = F.softmax(W, dim=1)
        row_weights = user_weights[user_idx]

        margins = (projected * row_weights).sum(dim=1) / LOGIT_SCALE

        # Sum of per-user mean Bradley-Terry negative log-likelihoods.
        # Softplus(-margin) = -log(sigmoid(margin)).
        loss = sum(
            F.softplus(-margins[s]).mean()
            for s in slices
        )

        loss.backward()
        optimizer.step()

        if (step + 1) % 250 == 0 or step == 0:
            print(
                f"Joint step {step + 1:4d}/{steps} | "
                f"loss = {loss.item():.5f}"
            )

    with torch.no_grad():
        W_probs = F.softmax(W, dim=1).detach()

    return {
        "V": V.detach(),
        "W": W_probs,
        "workers": worker_order,
        "final_loss": float(loss.item()),
    }


# --------------------------------------------------
# Adapt weights for unseen users, keeping V frozen
# --------------------------------------------------
def fit_lore_fewshot(diffs_by_user, workers, V, seed,
                     k=K, steps=FEWSHOT_STEPS, lr=FEWSHOT_LR):
    """
    Learn one simplex-constrained preference weight vector per unseen user.
    V remains frozen.
    """
    set_seed(seed + 100_000)

    X_cat, user_idx, slices, worker_order = pack_user_diffs(
        diffs_by_user, workers
    )

    n_users = len(worker_order)

    W_logits = nn.Parameter(torch.randn(n_users, k, device=DEVICE))
    optimizer = torch.optim.Adam([W_logits], lr=lr)

    V_fixed = V.detach()
    projected = X_cat @ V_fixed

    for step in range(steps):
        optimizer.zero_grad()

        user_weights = F.softmax(W_logits, dim=1)
        row_weights = user_weights[user_idx]

        margins = (projected * row_weights).sum(dim=1) / LOGIT_SCALE

        loss = sum(
            F.softplus(-margins[s]).mean()
            for s in slices
        )

        loss.backward()
        optimizer.step()

    with torch.no_grad():
        W_probs = F.softmax(W_logits, dim=1).detach()

    return {
        "W": W_probs,
        "workers": worker_order,
        "final_loss": float(loss.item()),
    }


# --------------------------------------------------
# Evaluate pairwise preference accuracy per user
# --------------------------------------------------
@torch.no_grad()
def evaluate_lore_users(diffs_by_user, workers, V, W):
    """
    Accuracy is the fraction of comparisons for which
    score(winning) > score(losing), equivalently margin > 0.

    Returns:
        dict: worker ID -> accuracy in [0, 1]
    """
    results = {}

    V = V.to(DEVICE)
    W = W.to(DEVICE)

    for i, worker in enumerate(workers):
        X = diffs_by_user[worker].to(DEVICE, dtype=torch.float32)
        margins = (X @ V @ W[i]) / LOGIT_SCALE
        results[worker] = (margins > 0).float().mean().item()

    return results


def summarize_user_accuracies(accuracy_dict):
    """
    Macro-average across users. This keeps users with many annotations
    from dominating the overall result.
    """
    values = np.array(list(accuracy_dict.values()), dtype=float)

    return {
        "mean": float(values.mean()),
        "std_across_users": float(values.std(ddof=0)),
        "n_users": int(len(values)),
    }


# --------------------------------------------------
# Quick checks: no training yet
# --------------------------------------------------
print("\nChecking tensor preparation...")

X_check, idx_check, slices_check, workers_check = pack_user_diffs(
    train_diffs, sorted(seen_users)
)

assert X_check.ndim == 2
assert X_check.shape[1] == 4096
assert len(idx_check) == X_check.shape[0]
assert len(slices_check) == len(seen_users)

print("Seen-user packed tensor:", tuple(X_check.shape))
print("Seen-user count:", len(workers_check))
print("Unseen-user count:", len(unseen_users))

# Verify the number of training comparisons available for each setting.
for n_examples in (100, 150):
    available = [
        min(len(train_diffs[u]), n_examples)
        for u in sorted(seen_users)
    ]
    print(
        f"Setting {n_examples}: "
        f"{sum(available)} total seen-user comparisons; "
        f"per-user range = {min(available)}–{max(available)}"
    )

print("\nAll checks passed. No training runs have started.")

Device: cuda
K: 5
Joint training: 1000 steps; lr = 0.5
Few-shot adaptation: 500 steps; lr = 0.1

Checking tensor preparation...
Seen-user packed tensor: (2915, 4096)
Seen-user count: 20
Unseen-user count: 19
Setting 100: 1984 total seen-user comparisons; per-user range = 84–100
Setting 150: 2915 total seen-user comparisons; per-user range = 84–150

All checks passed. No training runs have started.


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# --------------------------------------------------
# Configuration
# --------------------------------------------------
SEEDS = [42, 43, 44, 45, 46]
SEEN_EXAMPLE_SETTINGS = [100, 150]

OUTPUT_DIR = (
    REDDIT_DIR.parent
    / "RedditTLDR_runs"
    / "K5_1000joint_500fewshot_5seeds"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_PATH = OUTPUT_DIR / "run_metrics.csv"

seen_workers_ordered = sorted(seen_users)
unseen_workers_ordered = sorted(unseen_users)


# --------------------------------------------------
# Reproducible, nested sampling of seen-user examples
# For a given seed, the 100-example set is a subset
# of that seed's 150-example set for each user.
# --------------------------------------------------
def sample_seen_training_data(seed, target_count):
    sampled = {}

    for worker_idx, worker in enumerate(seen_workers_ordered):
        X = train_diffs[worker]
        n = min(target_count, len(X))

        rng = np.random.default_rng(seed + 1000 * worker_idx)
        permutation = rng.permutation(len(X))
        selected_indices = permutation[:n]

        sampled[worker] = X[selected_indices].clone()

    return sampled


# --------------------------------------------------
# Load completed results, if resuming
# --------------------------------------------------
if METRICS_PATH.exists():
    previous_df = pd.read_csv(METRICS_PATH)
else:
    previous_df = pd.DataFrame()

completed_ids = (
    set(previous_df["run_id"].astype(str))
    if not previous_df.empty and "run_id" in previous_df.columns
    else set()
)

print("Output directory:", OUTPUT_DIR)
print("Already recorded runs:", len(completed_ids))


# --------------------------------------------------
# Execute experiments
# --------------------------------------------------
for target_count in SEEN_EXAMPLE_SETTINGS:
    for seed in SEEDS:

        run_id = f"nseen{target_count}_seed{seed}"
        checkpoint_path = OUTPUT_DIR / f"{run_id}.pt"

        if run_id in completed_ids and checkpoint_path.exists():
            print(f"\nSkipping completed run: {run_id}")
            continue

        print("\n" + "=" * 70)
        print(f"RUN: {run_id}")
        print("=" * 70)

        # 1. Select seen-user training comparisons.
        seen_train_subset = sample_seen_training_data(
            seed=seed,
            target_count=target_count,
        )

        actual_seen_pairs = sum(
            len(X) for X in seen_train_subset.values()
        )

        print("Seen users:", len(seen_workers_ordered))
        print("Unseen users:", len(unseen_workers_ordered))
        print("Seen-user training comparisons:", actual_seen_pairs)
        print("Few-shot comparisons per unseen user: 50")

        # 2. Learn shared basis and seen-user weights.
        joint_result = fit_lore_joint(
            diffs_by_user=seen_train_subset,
            workers=seen_workers_ordered,
            seed=seed,
            k=K,
            steps=JOINT_STEPS,
            lr=JOINT_LR,
        )

        V = joint_result["V"]
        W_seen = joint_result["W"]

        # 3. Adapt weights for unseen users using all 50
        # cached few-shot comparisons per user.
        fewshot_result = fit_lore_fewshot(
            diffs_by_user=train_diffs,
            workers=unseen_workers_ordered,
            V=V,
            seed=seed,
            k=K,
            steps=FEWSHOT_STEPS,
            lr=FEWSHOT_LR,
        )

        W_unseen = fewshot_result["W"]

        # 4. Measure seen training and held-out accuracy.
        seen_train_user_acc = evaluate_lore_users(
            diffs_by_user=seen_train_subset,
            workers=seen_workers_ordered,
            V=V,
            W=W_seen,
        )

        seen_val_user_acc = evaluate_lore_users(
            diffs_by_user=val_diffs,
            workers=seen_workers_ordered,
            V=V,
            W=W_seen,
        )

        # 5. Measure unseen few-shot training and held-out accuracy.
        unseen_train_user_acc = evaluate_lore_users(
            diffs_by_user=train_diffs,
            workers=unseen_workers_ordered,
            V=V,
            W=W_unseen,
        )

        unseen_val_user_acc = evaluate_lore_users(
            diffs_by_user=val_diffs,
            workers=unseen_workers_ordered,
            V=V,
            W=W_unseen,
        )

        seen_train_summary = summarize_user_accuracies(
            seen_train_user_acc
        )
        seen_val_summary = summarize_user_accuracies(
            seen_val_user_acc
        )
        unseen_train_summary = summarize_user_accuracies(
            unseen_train_user_acc
        )
        unseen_val_summary = summarize_user_accuracies(
            unseen_val_user_acc
        )

        # Give seen and unseen groups equal weight, consistent
        # with the paper's overall metric.
        overall_val_acc = (
            seen_val_summary["mean"] + unseen_val_summary["mean"]
        ) / 2

        # 6. Save checkpoint, storing tensors on CPU.
        torch.save(
            {
                "run_id": run_id,
                "seed": seed,
                "target_seen_examples": target_count,
                "actual_seen_training_comparisons": actual_seen_pairs,
                "K": K,
                "V": V.detach().cpu(),
                "W_seen": W_seen.detach().cpu(),
                "W_unseen": W_unseen.detach().cpu(),
                "seen_workers": seen_workers_ordered,
                "unseen_workers": unseen_workers_ordered,
                "joint_loss": joint_result["final_loss"],
                "fewshot_loss": fewshot_result["final_loss"],
            },
            checkpoint_path,
        )

        # 7. Record metrics for this run.
        row = {
            "run_id": run_id,
            "seed": seed,
            "target_seen_examples": target_count,
            "actual_seen_training_comparisons": actual_seen_pairs,
            "K": K,
            "joint_steps": JOINT_STEPS,
            "joint_lr": JOINT_LR,
            "fewshot_steps": FEWSHOT_STEPS,
            "fewshot_lr": FEWSHOT_LR,
            "seen_train_accuracy": seen_train_summary["mean"],
            "seen_validation_accuracy": seen_val_summary["mean"],
            "seen_validation_std_across_users":
                seen_val_summary["std_across_users"],
            "unseen_fewshot_train_accuracy":
                unseen_train_summary["mean"],
            "unseen_validation_accuracy": unseen_val_summary["mean"],
            "unseen_validation_std_across_users":
                unseen_val_summary["std_across_users"],
            "overall_validation_accuracy": overall_val_acc,
            "joint_final_loss": joint_result["final_loss"],
            "fewshot_final_loss": fewshot_result["final_loss"],
        }

        # Save after every run, so completed work is not lost.
        previous_df = pd.concat(
            [previous_df, pd.DataFrame([row])],
            ignore_index=True,
        )
        previous_df.to_csv(METRICS_PATH, index=False)
        completed_ids.add(run_id)

        print(f"\nRun completed: {run_id}")
        print(f"Seen validation accuracy:   "
              f"{100 * row['seen_validation_accuracy']:.2f}%")
        print(f"Unseen validation accuracy: "
              f"{100 * row['unseen_validation_accuracy']:.2f}%")
        print(f"Overall validation accuracy: "
              f"{100 * row['overall_validation_accuracy']:.2f}%")
        print("Checkpoint saved:", checkpoint_path.name)
        print("Metrics updated:", METRICS_PATH)


print("\n" + "=" * 70)
print("EXPERIMENT SWEEP FINISHED")
print("=" * 70)

if METRICS_PATH.exists():
    results = pd.read_csv(METRICS_PATH)
    print(
        results.groupby("target_seen_examples")["run_id"]
        .count()
        .rename("completed_runs")
    )
    display(results.sort_values(["target_seen_examples", "seed"]))

print("\nOutput folder:", OUTPUT_DIR)

Output directory: /content/drive/MyDrive/LoRe/RedditTLDR_runs/K5_1000joint_500fewshot_5seeds
Already recorded runs: 0

RUN: nseen100_seed42
Seen users: 20
Unseen users: 19
Seen-user training comparisons: 1984
Few-shot comparisons per unseen user: 50
Joint step    1/1000 | loss = 14.24719
Joint step  250/1000 | loss = 0.29113
Joint step  500/1000 | loss = 0.12923
Joint step  750/1000 | loss = 0.08352
Joint step 1000/1000 | loss = 0.06317

Run completed: nseen100_seed42
Seen validation accuracy:   64.79%
Unseen validation accuracy: 65.13%
Overall validation accuracy: 64.96%
Checkpoint saved: nseen100_seed42.pt
Metrics updated: /content/drive/MyDrive/LoRe/RedditTLDR_runs/K5_1000joint_500fewshot_5seeds/run_metrics.csv

RUN: nseen100_seed43
Seen users: 20
Unseen users: 19
Seen-user training comparisons: 1984
Few-shot comparisons per unseen user: 50
Joint step    1/1000 | loss = 13.79679
Joint step  250/1000 | loss = 0.25643
Joint step  500/1000 | loss = 0.10133
Joint step  750/1000 | loss =

,run_id,seed,target_seen_examples,actual_seen_training_comparisons,K,joint_steps,joint_lr,fewshot_steps,fewshot_lr,seen_train_accuracy,seen_validation_accuracy,seen_validation_std_across_users,unseen_fewshot_train_accuracy,unseen_validation_accuracy,unseen_validation_std_across_users,overall_validation_accuracy,joint_final_loss,fewshot_final_loss
0,nseen100_seed42,42,100,1984,5,1000,0.5,500,0.1,0.999000,0.647852,0.028711,0.728421,0.651270,0.078487,0.649561,0.063174,17.707506
1,nseen100_seed43,43,100,1984,5,1000,0.5,500,0.1,1.000000,0.646240,0.044888,0.722105,0.653294,0.058956,0.649767,0.036119,16.604750
2,nseen100_seed44,44,100,1984,5,1000,0.5,500,0.1,0.998500,0.645729,0.034766,0.738947,0.654085,0.066554,0.649907,0.091502,16.841690
3,nseen100_seed45,45,100,1984,5,1000,0.5,500,0.1,0.998000,0.643255,0.041527,0.711579,0.664106,0.084907,0.653680,0.091150,16.062519
4,nseen100_seed46,46,100,1984,5,1000,0.5,500,0.1,0.999000,0.626735,0.040445,0.724211,0.641021,0.081081,0.633878,0.074073,16.927935
5,nseen150_seed42,42,150,2915,5,1000,0.5,500,0.1,0.997333,0.640583,0.028791,0.735789,0.650301,0.105568,0.645442,0.141774,16.960487
6,nseen150_seed43,43,150,2915,5,1000,0.5,500,0.1,0.997333,0.632752,0.040408,0.714737,0.657866,0.064915,0.645309,0.176872,17.305882
7,nseen150_seed44,44,150,2915,5,1000,0.5,500,0.1,0.999000,0.647983,0.032853,0.726316,0.651673,0.056178,0.649828,0.118971,17.572737
8,nseen150_seed45,45,150,2915,5,1000,0.5,500,0.1,0.997667,0.643855,0.034822,0.733684,0.662656,0.083360,0.653256,0.141089,16.433393
9,nseen150_seed46,46,150,2915,5,1000,0.5,500,0.1,0.998000,0.636530,0.031232,0.731579,0.650098,0.056157,0.643314,0.157595,16.458294



Output folder: /content/drive/MyDrive/LoRe/RedditTLDR_runs/K5_1000joint_500fewshot_5seeds


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# Load and validate run results
# --------------------------------------------------
df = pd.read_csv(METRICS_PATH)

# Guard against duplicate rows if a run was ever recorded twice.
df = df.drop_duplicates(subset="run_id", keep="last")

expected_runs = {
    (n, seed)
    for n in [100, 150]
    for seed in [42, 43, 44, 45, 46]
}

actual_runs = set(
    zip(
        df["target_seen_examples"].astype(int),
        df["seed"].astype(int),
    )
)

missing_runs = expected_runs - actual_runs
if missing_runs:
    raise ValueError(f"Missing experiment runs: {missing_runs}")

# Paper Table 2: Reddit TL;DR LoRe results, in percent.
paper = {
    100: {
        "seen_validation_accuracy": (65.0, 1.1),
        "unseen_validation_accuracy": (66.2, 1.2),
        "overall_validation_accuracy": (65.6, 1.1),
    },
    150: {
        "seen_validation_accuracy": (66.2, 1.0),
        "unseen_validation_accuracy": (66.7, 1.1),
        "overall_validation_accuracy": (66.5, 1.0),
    },
}

metrics = {
    "seen_validation_accuracy": "Seen-user validation",
    "unseen_validation_accuracy": "Unseen-user validation",
    "overall_validation_accuracy": "Overall validation",
}


# --------------------------------------------------
# Aggregate across seeds and compare with paper
# --------------------------------------------------
summary_rows = []
comparison_rows = []

for setting in [100, 150]:
    group = df[df["target_seen_examples"] == setting].sort_values("seed")

    assert len(group) == 5, (
        f"Expected five runs for setting {setting}, got {len(group)}"
    )

    summary_row = {
        "seen_examples_per_user": setting,
        "number_of_seeds": len(group),
    }

    for metric, label in metrics.items():
        values_pct = group[metric].astype(float) * 100

        mean_pct = values_pct.mean()
        std_pct = values_pct.std(ddof=1)  # Sample SD across five seeds

        paper_mean, paper_std = paper[setting][metric]
        delta = mean_pct - paper_mean

        summary_row[f"{metric}_mean_pct"] = mean_pct
        summary_row[f"{metric}_seed_sd_pp"] = std_pct

        comparison_rows.append({
            "seen_examples_per_user": setting,
            "metric": label,
            "ours_mean_pct": mean_pct,
            "ours_seed_sd_pp": std_pct,
            "paper_mean_pct": paper_mean,
            "paper_reported_sd_pp": paper_std,
            "difference_vs_paper_pp": delta,
        })

    summary_rows.append(summary_row)


summary_df = pd.DataFrame(summary_rows)
comparison_df = pd.DataFrame(comparison_rows)

# --------------------------------------------------
# Save outputs
# --------------------------------------------------
SUMMARY_PATH = OUTPUT_DIR / "five_seed_summary.csv"
COMPARISON_PATH = OUTPUT_DIR / "paper_vs_ours.csv"

summary_df.to_csv(SUMMARY_PATH, index=False)
comparison_df.to_csv(COMPARISON_PATH, index=False)

# --------------------------------------------------
# Display readable results
# --------------------------------------------------
print("=" * 85)
print("FIVE-SEED SUMMARY")
print("Mean accuracy ± sample standard deviation across seeds")
print("=" * 85)

for _, row in summary_df.iterrows():
    setting = int(row["seen_examples_per_user"])
    print(f"\nSetting: {setting} seen-user examples")
    for metric, label in metrics.items():
        mean = row[f"{metric}_mean_pct"]
        sd = row[f"{metric}_seed_sd_pp"]
        print(f"  {label:<26}: {mean:.2f}% ± {sd:.2f} pp")

print("\n" + "=" * 85)
print("PAPER VS OUR RESULTS")
print("Difference = our mean - paper mean (percentage points)")
print("=" * 85)

display(
    comparison_df.style.format({
        "ours_mean_pct": "{:.2f}%",
        "ours_seed_sd_pp": "{:.2f}",
        "paper_mean_pct": "{:.1f}%",
        "paper_reported_sd_pp": "{:.1f}",
        "difference_vs_paper_pp": "{:+.2f}",
    })
)

print("\nSaved:")
print(SUMMARY_PATH)
print(COMPARISON_PATH)

FIVE-SEED SUMMARY
Mean accuracy ± sample standard deviation across seeds

Setting: 100 seen-user examples
  Seen-user validation      : 64.20% ± 0.87 pp
  Unseen-user validation    : 65.28% ± 0.82 pp
  Overall validation        : 64.74% ± 0.77 pp

Setting: 150 seen-user examples
  Seen-user validation      : 64.03% ± 0.60 pp
  Unseen-user validation    : 65.45% ± 0.55 pp
  Overall validation        : 64.74% ± 0.40 pp

PAPER VS OUR RESULTS
Difference = our mean - paper mean (percentage points)


,seen_examples_per_user,metric,ours_mean_pct,ours_seed_sd_pp,paper_mean_pct,paper_reported_sd_pp,difference_vs_paper_pp
0,100,Seen-user validation,64.20%,0.87,65.0%,1.1,-0.80
1,100,Unseen-user validation,65.28%,0.82,66.2%,1.2,-0.92
2,100,Overall validation,64.74%,0.77,65.6%,1.1,-0.86
3,150,Seen-user validation,64.03%,0.60,66.2%,1.0,-2.17
4,150,Unseen-user validation,65.45%,0.55,66.7%,1.1,-1.25
5,150,Overall validation,64.74%,0.40,66.5%,1.0,-1.76



Saved:
/content/drive/MyDrive/LoRe/RedditTLDR_runs/K5_1000joint_500fewshot_5seeds/five_seed_summary.csv
/content/drive/MyDrive/LoRe/RedditTLDR_runs/K5_1000joint_500fewshot_5seeds/paper_vs_ours.csv
